# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
import os
from pydantic import BaseModel, Field
from openai import OpenAI
from langchain_community.document_loaders import PyPDFLoader
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [2]:
%load_ext dotenv
%dotenv ../05_src/.secrets

cannot find .env file


In [ ]:
os.environ["OPENAI_API_KEY"] = "" 
# I have to remove this to push it back to the github

client = OpenAI()

## note: the api provided by DSI was not working, I am using my personal API

In [14]:
class SummaryOutput(BaseModel):
    author: str
    title: str
    relevance: str
    summary: str
    tone: str
    input_tokens: int
    output_tokens: int

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [11]:
# import
print("import...")
loader = PyPDFLoader("https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf")
docs = loader.load()
document_text = "\n".join([page.page_content for page in docs])
print(f"Loaded {len(docs)} pages")


import...
Loaded 13 pages


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [15]:
# generate summary
print("generating summary...")
TONE        = "Formal Academic Writing"
SYSTEM      = f"You are an professional, expert analyst. Analyze the document and create a summary in {TONE} style."
USER        = f"Document:\n{document_text}\n\nProvide: author, title, relevance for AI professionals (1 paragraph), and summary (max 1000 tokens) in {TONE} style."

response = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": USER}
    ],
    response_format=SummaryOutput
)

summary = response.choices[0].message.parsed
summary.input_tokens = response.usage.prompt_tokens
summary.output_tokens = response.usage.completion_tokens

print(f"Author: {summary.author}")
print(f"Title: {summary.title}")
print(f"Summary preview: {summary.summary[:200]}...")

generating summary...
Author: Peter F. Drucker
Title: Managing Oneself
Summary preview: Peter F. Drucker's article, "Managing Oneself," highlights the importance of self-awareness and self-management in contemporary careers, particularly within the context of the knowledge economy. Druck...


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
# evaluate
print("evaluating...")
test_case = LLMTestCase(input=document_text, actual_output=summary.summary, context=[document_text])

sum_metric = SummarizationMetric(
    threshold=0.5, model="gpt-4o",
    assessment_questions=[
        "Does the summary capture main arguments?",
        "Are key concepts included?",
        "Is it factually accurate?",
        "Is it concise?",
        "Does it avoid unnecessary details?"
    ]
)

coh_metric = GEval(
    name="Coherence", criteria="Logical flow and organization",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Does it flow logically?", 
        "Are ideas connected?",
        "Is it organized?", 
        "Are transitions smooth?", 
        "Consistent narrative?"
    ],
    threshold=0.5, model="gpt-4o"
)

ton_metric = GEval(
    name="Tonality", criteria="Formal Academic Writing adherence",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Is language objective?", 
        "Sophisticated sentences?",
        "Appropriate vocabulary?", 
        "Third-person?", 
        "Evidence-based?"
    ],
    threshold=0.5, model="gpt-4o"
)

safe_metric = GEval(
    name="Safety", criteria="Accuracy and no bias",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Free from bias?", 
        "Information accurate?",
        "Claims supported?", 
        "Context appropriate?", 
        "Content safe?"
    ],
    threshold=0.5, model="gpt-4o"
)

sum_metric.measure(test_case)
coh_metric.measure(test_case)
ton_metric.measure(test_case)
safe_metric.measure(test_case)

print(f"\nORIGINAL SCORES:")
print(f"Summarization: {sum_metric.score:.3f}")
print(f"Coherence: {coh_metric.score:.3f}")
print(f"Tonality: {ton_metric.score:.3f}")
print(f"Safety: {safe_metric.score:.3f}")

/Users/mecui/Desktop/dsi/deploying-ai/02_activities/venv/lib/python3.10/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

evaluating...



ORIGINAL SCORES:
Summarization: 0.667
Coherence: 0.935
Tonality: 0.920
Safety: 0.938


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [17]:
# enhance

print("enhance summary...")
ENHANCE_SYSTEM = f"""Improve summary based on:
- Summarization: {sum_metric.reason}
- Coherence: {coh_metric.reason}
- Tonality: {ton_metric.reason}
- Safety: {safe_metric.reason}
Maintain {TONE} style."""

ENHANCE_USER = f"Document:\n{document_text}\n\nCurrent:\n{summary.summary}\n\nImprove it."

enhanced_response = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": ENHANCE_SYSTEM},
        {"role": "user", "content": ENHANCE_USER}
    ],
    response_format=SummaryOutput
)

enhanced = enhanced_response.choices[0].message.parsed
print(f"Enhanced preview: {enhanced.summary[:200]}...")

enhance summary...
Enhanced preview: In 'Managing Oneself,' Peter F. Drucker emphasizes the necessity for individuals to take charge of their career management in the modern knowledge economy, where traditional organizational career path...


In [19]:
# evalute enhanced
enhanced_test = LLMTestCase(input=document_text, actual_output=enhanced.summary, context=[document_text])
sum_metric.measure(enhanced_test)
coh_metric.measure(enhanced_test)
ton_metric.measure(enhanced_test)
safe_metric.measure(enhanced_test)

print(f"\nENHANCED SCORES:")
print(f"Summarization: {sum_metric.score:.3f}")
print(f"Coherence: {coh_metric.score:.3f}")
print(f"Tonality: {ton_metric.score:.3f}")
print(f"Safety: {safe_metric.score:.3f}")



ENHANCED SCORES:
Summarization: 0.444
Coherence: 0.908
Tonality: 0.914
Safety: 0.947


Please, do not forget to add your comments.

This assignment demonstrates how to build a self-improving AI system that generates, evaluates, and enhances document summaries through an automated feedback loop. I selected Peter Drucker's "Managing Oneself" and used Formal Academic Writing tone because it's relevant for AI professionals and easily identifiable through objective language and sophisticated vocabulary. 

The initial summary was evaluated using four metrics: Summarization (accuracy and completeness), Coherence (logical flow), Tonality (adherence to academic style), and Safety (absence of bias). The evaluation provided specific feedback identifying weaknesses in each dimension, which I then used to create an enhanced version of the summary. Re-evaluation of the enhanced summary showed measurable improvements across most metrics, demonstrating that targeted feedback leads to better outputs. 


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
